# 12 Cross-simulation check: the TNG model on Bolshoi

The packaged model was trained on TNG300-3-Dark subhalos only. Here it predicts
the whole Bolshoi box from Bolshoi halos at the same four mean separations,
and is scored against Bolshoi's own T-web labels (`bolshoi/mask`, effective
smoothing $1\,h^{-1}\,\mathrm{Mpc}$, threshold 0.65, as for TNG). Nothing
from Bolshoi was used in training, so the whole box is a test set.

What differs between the two boxes, all at once:

- cosmology (TNG: Planck 2015; Bolshoi: WMAP-like) and box size (205 vs
  $250\,h^{-1}\,\mathrm{Mpc}$);
- cell size: 0.400 vs $0.391\,h^{-1}\,\mathrm{Mpc}$ (2.5% smaller);
- tracers: Subfind subhalos vs BDM halos. Check how the Bolshoi halo grids
  were weighted (mass or counts) before reading too much into the scores; the
  TNG inputs are mass-weighted.

Two ways of scaling the inputs are compared (see `deepvoid.pretrained`):
`saved` uses the TNG statistics of the matching spacing (the default),
`refit` fits them on the Bolshoi box itself.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/deepvoid
!git clone -q https://github.com/skumagai3/deepvoid.git /content/deepvoid
!pip -q install --force-reinstall --no-deps /content/deepvoid
!git -C /content/deepvoid log --oneline -1

In [ ]:
import json
import numpy as np
import pandas as pd
from deepvoid import Catalog
from deepvoid.evaluate import confusion_matrix, scores
from deepvoid.pretrained import Pretrained

MODEL = '/content/drive/MyDrive/deepvoid_model_release'   # from notebook 11
model = Pretrained.from_folder(MODEL)
catalog = Catalog()
truth = catalog.load('bolshoi/mask').data
print(model.info['name'], 'cell size', model.info['cell_size'], 'spacings', model.spacings)

## 1. Predict and score the whole Bolshoi box

In [ ]:
KEYS = ['mcc', 'void_mcc', 'void_f1', 'wall_f1', 'filament_f1', 'halo_f1']
rows, cms = {}, {}
for s in (3, 5, 7, 10):
    vol = catalog.load(f'bolshoi/halos/L{s}')
    for stats in ('saved', 'refit'):
        pred = model.predict(vol.data, vol.box_size, float(s), stats=stats).argmax(-1)
        cm = confusion_matrix(truth, pred)
        cms[(s, stats)] = cm
        rows[(s, stats)] = {k: scores(cm)[k] for k in KEYS}
        print(s, stats, {k: round(v, 3) for k, v in rows[(s, stats)].items()}, flush=True)
    del vol, pred
bolshoi = pd.DataFrame(rows).T
bolshoi.index.names = ['spacing', 'stats']

## 2. Against the TNG test slab

In [ ]:
tng = json.load(open(f'{MODEL}/test_scores.json'))
tng = pd.DataFrame({int(k.rsplit('L', 1)[1]): {m: v[m] for m in KEYS} for k, v in tng.items()}).T
table = pd.concat({'TNG test slab': tng,
                   'Bolshoi, saved': bolshoi.xs('saved', level='stats'),
                   'Bolshoi, refit': bolshoi.xs('refit', level='stats')}, axis=1)
print(table.round(3).to_string())

## 3. Save

Scores and confusion matrices go next to the model so they can be read later.

In [ ]:
out = {f'L{s} {stats}': {'scores': rows[(s, stats)], 'confusion_matrix': cms[(s, stats)].tolist()}
       for (s, stats) in rows}
path = '/content/drive/MyDrive/deepvoid_runs/cross_simulation_bolshoi.json'
json.dump(out, open(path, 'w'), indent=1)
print('saved', path)